# Bi-encoder FT-v4 — continue from FT-300 with all-types augmentation

Base: included `ft-300-base/` (sentence-transformers MiniLM-L6 + pooling + normalize, 87 MB)
Loss: MultipleNegativesRankingLoss (in-batch negatives)
Train: 5622 (q, gold_doc) positive pairs (100 LME train q + 5448 syn paraphrase)
Val: 293 pairs

**Kaggle:** Add Input → `chat-ce-bi-ftv4` dataset. Accelerator → **GPU T4 x2**. Run All. Süre: ~5-10 dk.

In [ ]:
# 1) Setup
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers==5.4.1', 'torch'])
import torch
print('cuda?', torch.cuda.is_available(), 'device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

In [ ]:
# 2) Locate data + base model
from pathlib import Path
import glob
found = glob.glob('/kaggle/input/**/s3_biencoder_pairs.jsonl', recursive=True)
assert found, 'train pairs not found in /kaggle/input/. Attach dataset.'
TRAIN = Path(found[0]); VAL = TRAIN.parent / 's3_biencoder_pairs_val.jsonl'
# locate ft-300 base inside same dataset dir
base_cands = glob.glob('/kaggle/input/**/ft-300-base/config.json', recursive=True)
assert base_cands, 'ft-300-base/ not found. Upload it as part of the dataset.'
FT300 = str(Path(base_cands[0]).parent)
print('train:', TRAIN); print('val:  ', VAL); print('base: ', FT300)

In [ ]:
# 3) Load examples
import json
from sentence_transformers import InputExample
def load(p):
    out = []
    for line in open(p):
        r = json.loads(line)
        out.append(InputExample(texts=[r['q'], r['doc']]))
    return out
train_ex = load(TRAIN); val_ex = load(VAL)
print(f'train={len(train_ex)} val={len(val_ex)}')

In [ ]:
# 4) Train (MNR loss, continue from FT-300)
from sentence_transformers import SentenceTransformer, losses
from torch.utils.data import DataLoader
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
OUT = './ft-v4-20260516'
EPOCHS = 2; BATCH = 64; LR = 2e-5

model = SentenceTransformer(FT300)
loader = DataLoader(train_ex, shuffle=True, batch_size=BATCH)
loss = losses.MultipleNegativesRankingLoss(model)
warmup = int(0.1 * len(loader) * EPOCHS)
print(f'epochs={EPOCHS} batch={BATCH} lr={LR} steps/epoch={len(loader)} warmup={warmup}')
model.fit(
    train_objectives=[(loader, loss)],
    epochs=EPOCHS,
    warmup_steps=warmup,
    optimizer_params={'lr': LR},
    output_path=OUT,
    show_progress_bar=True,
)
model.save(OUT)
print('saved files:', sorted(os.listdir(OUT)))

In [ ]:
# 5) Val sanity — cosine of (q, gold_doc) pairs should average high
import numpy as np
from sentence_transformers import SentenceTransformer as ST
m = ST(OUT)
vq = [ex.texts[0] for ex in val_ex]; vd = [ex.texts[1] for ex in val_ex]
eq = m.encode(vq, batch_size=64, convert_to_numpy=True, normalize_embeddings=True)
ed = m.encode(vd, batch_size=64, convert_to_numpy=True, normalize_embeddings=True)
cos_pos = (eq * ed).sum(axis=1)
# random negative baseline
perm = np.random.permutation(len(vd))
cos_neg = (eq * ed[perm]).sum(axis=1)
print(f'val positive cosine mean: {cos_pos.mean():.4f}  (n={len(cos_pos)})')
print(f'val random-neg  cos mean: {cos_neg.mean():.4f}')
print(f'margin (pos - neg): {(cos_pos.mean() - cos_neg.mean()):.4f}')

In [ ]:
# 6) Pack + FileLink
import shutil
shutil.make_archive('ft-v4', 'zip', OUT)
print('size:', os.path.getsize('ft-v4.zip')//1024, 'KB')
from IPython.display import FileLink
FileLink('ft-v4.zip')